# 3.4 The impacts of WMH on CM-FC coupling and memory deficits
## Mediation analysis

In [1]:

from brainspace.plotting import plot_hemispheres
from brainspace.utils.parcellation import map_to_labels
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import scipy as sp
import nilearn as nil
from nilearn import plotting
from scipy.stats import f_oneway
import statsmodels.api as sm
import seaborn as sns
from matplotlib.ticker import StrMethodFormatter
import os
import nibabel as nib
from brainspace.mesh.mesh_io import read_surface
from netneurotools.metrics import *
from statsmodels.stats.multitest import multipletests

from PIL import Image, ImageDraw, ImageFont
import pingouin as pg
from scipy.stats import pearsonr
from ana_utils import *
import warnings
warnings.filterwarnings("ignore")

d:\ProgramFiles\Anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
d:\ProgramFiles\Anaconda3\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
d:\ProgramFiles\Anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
output_dir = f"{PROJ_HOME}/results/4_mediation_analysis"
os.makedirs(output_dir, exist_ok=True)
print("Output dir:", output_dir)

Output dir: D:\Shanghaitec\PROJECT\CSVD_neo/results/4_mediation_analysis


In [ ]:
roi_lab = "nawm"
med_items = ["rd", "fa", "adc", "ad", "logwmh"]
#  "rd", "fa","adc", "ad","tfr","tfr",

dwi_metric_csv = f"{PROJ_HOME}/results/lms_stat/4_wmh_dwi_metric.csv"
cog_items = [
    "MMSE",
    "MoCA",
    "TMT(B-A)",
    "Stroop A-T",
    "Stroop B-T",
    "Stroop C-T",
    "Stroop-TI",
    "Stroop-EI",
    "DS(f+b)",
    "DS-forward",
    "DS-backward",
    "DSS",
    "VFT",
    "AVLT1",
    "AVLT2",
    "AVLT3",
    "AVLT4",
    "AVLT5",
    "AVLT再认",
    "AVLT(1+2+3)",
    "Rey-O回忆",
    "Rey-O再认",
    "Rey-O复制",
    "BNT",
]


In [4]:
# Get sub_info tbl
sub_info_tbl = pd.read_csv(f"{PROJ_HOME}/data/filt_sub_info.csv", encoding="gbk")

sub_info_tbl["Stroop-TI"] = (
    sub_info_tbl["Stroop C-T"]
    - (sub_info_tbl["Stroop A-T"] + sub_info_tbl["Stroop B-T"]) * 0.5
)
sub_info_tbl["Stroop-EI"] = (
    50-sub_info_tbl["Stroop C-N"]
    - (50-sub_info_tbl["Stroop A-N"] + 50-sub_info_tbl["Stroop B-N"]) * 0.5
)


mean_edu_year = sub_info_tbl["edu_year"].mean().round()
# Fill education year nan to mean
sub_info_tbl["edu_year"].fillna(mean_edu_year, inplace=True)

# Get NAWM volume
nawm_tbl = pd.read_csv(
    f"{PROJ_HOME}/results/lms_stat/nawm_volume_raw.csv", names=["pid", "nawm"]
)
nawm_tbl["lognawm"] = nawm_tbl["nawm"].apply(lambda x: np.log(x))
sub_info_tbl = pd.merge(sub_info_tbl, on="pid", right=nawm_tbl)

roi_tbl = pd.read_csv(
    dwi_metric_csv,
    names=["pid", "dwim", "val"],
)
wmh_met_pivot = roi_tbl.pivot_table(
    index="pid", values="val", columns="dwim"
).reset_index()
sub_info_tbl = pd.merge(sub_info_tbl, on="pid", right=wmh_met_pivot)
print(sub_info_tbl.columns)

Index(['Unnamed: 0', 'ID', 'pid', 'isenrolled', 'hasfc', 'other_info',
       '姓名（拼音）', '姓名（中文）', 'namecount', 'age',
       ...
       'log_wmh_vol', 'genderv', 'Stroop-TI', 'Stroop-EI', 'nawm', 'lognawm',
       'ad', 'adc', 'fa', 'rd'],
      dtype='object', length=136)


In [5]:
def check_indirect_eff(df):
    df_pos = df[df.sig=="Yes"]
    # df.path contain "Direct" and "Indirect *", then return True
    paths = df_pos['path'].values.tolist()
    if "Direct" in paths:
        if any("Indirect" in p for p in paths):
            return True
        else:
            return False

In [6]:
med_res_df = pd.DataFrame([])
sum_res = pd.DataFrame()
for med_item in med_items:
    # Global mediation analysis
    for cog_item in cog_items:
        for tract_lab in TRACT_LABS:
            
            print(cog_item, tract_lab, med_item)
            df = pd.read_csv(
                f"{PROJ_HOME}/results/1_global_CFC/_lr-global_cfc-{tract_lab}.csv"
            )
            df["age"] = df["pid"].map(SUB_ID_AGE_DICT)
            df["edu_year"] = df["pid"].map(
                sub_info_tbl.set_index("pid")["edu_year"].to_dict()
            )
            df["cog"] = df["pid"].map(
                sub_info_tbl.set_index("pid")[cog_item].to_dict()
            )
            df[med_item] = df["pid"].map(
                sub_info_tbl.set_index("pid")[med_item].to_dict()
            )
            df = df[~np.isnan(df["cog"])]
            df = df[df[med_item] > 0]
            pg_ma_res = pg.mediation_analysis(
                data=df,
                x=med_item,
                m=CM_LABS,
                y="cog",
                covar=["age", "edu_year"],
                alpha=0.05,
                n_boot=1000,
                seed=42,
            )
            med_res_df = pd.concat(
                [
                    med_res_df,
                    pg_ma_res.assign(
                        Cognition=cog_item,
                        Tractography=tract_lab,
                        WMH_feature=med_item,
                        is_significant=check_indirect_eff(pg_ma_res)
                    ),
                ],
                ignore_index=True,
            )
med_res_df.to_csv(f"{output_dir}/mediation_analysis_global_cfc.csv", index=False)

AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb adc
AVLT(1+2+3) int adc
AVLT(1+2+3) wmh adc
AVLT(1+2+3) wb ad
AVLT(1+2+3) int ad
AVLT(1+2+3) wmh ad
AVLT(1+2+3) wb logwmh
AVLT(1+2+3) int logwmh
AVLT(1+2+3) wmh logwmh


In [7]:
med_res_df = pd.DataFrame([])
sum_res = pd.DataFrame()
for med_item in med_items:
    # Global mediation analysis
    for cog_item in cog_items:
        for net_lab in NET_LABS:
            for tract_lab in TRACT_LABS:
                
                print(cog_item, tract_lab, med_item)
                df = pd.read_csv(
                    f"{PROJ_HOME}/results/2_local_CFC/_lr-local_cfc-{tract_lab}.csv"
                )
                df = df[df['net']==net_lab]
                df["age"] = df["pid"].map(SUB_ID_AGE_DICT)
                df["edu_year"] = df["pid"].map(
                    sub_info_tbl.set_index("pid")["edu_year"].to_dict()
                )
                df["cog"] = df["pid"].map(
                    sub_info_tbl.set_index("pid")[cog_item].to_dict()
                )
                df[med_item] = df["pid"].map(
                    sub_info_tbl.set_index("pid")[med_item].to_dict()
                )
                df = df[~np.isnan(df["cog"])]
                df = df[df[med_item] > 0]
                pg_ma_res = pg.mediation_analysis(
                    data=df,
                    x=med_item,
                    m=CM_LABS,
                    y="cog",
                    covar=["age", "edu_year"],
                    alpha=0.05,
                    n_boot=1000,
                    seed=42,
                )
                med_res_df = pd.concat(
                    [
                        med_res_df,
                        pg_ma_res.assign(
                            Cognition=cog_item,
                            Tractography=tract_lab,
                            Network=net_lab,
                            WMH_feature=med_item,
                            is_significant=check_indirect_eff(pg_ma_res)
                        ),
                    ],
                    ignore_index=True,
                )
med_res_df.to_csv(f"{output_dir}/mediation_analysis_local_cfc.csv", index=False)

AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb rd
AVLT(1+2+3) int rd
AVLT(1+2+3) wmh rd
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb fa
AVLT(1+2+3) int fa
AVLT(1+2+3) wmh fa
AVLT(1+2+3) wb adc
AVLT(1+2+3) int adc
AVLT(1+2+3) wmh adc
AVLT(1+2+3) wb adc
AVLT(1+2+3) int adc
AVLT(1+2+3) wmh adc
AVLT(1+2+3) wb adc
AVLT(1+2+3) int adc
AVLT(1+2+3) wmh adc
AVLT(1+2+3) wb adc
AVLT(1+2+3) int adc


In [8]:
stop here

SyntaxError: invalid syntax (4067800170.py, line 1)

In [ ]:

for cog_item in ["门诊MMSE", "MoCA-merge"]:
    print(cog_item)
    for tg_lab in TRACT_LABS:
        for cm in CM_ARR:
            cm_name, cm_idx, _ = cm
            print(cog_item, tg_lab,cm_name)
            df = pd.read_csv(f"{PROJ_HOME}/results/lr_net_lm_cfc/{cm_name}-cfc_{tg_lab}.csv")
            
            df = df[df.idx ==cm_idx]
            
            df = pd.merge(df, right=sub_info_tbl.set_index("pid")[["age", "edu_year", cog_item, med_item]],on="pid")
            
            df = df[~np.isnan(df[cog_item])]
            # df= df[df[med_item]>0]
            
            df = df.pivot_table(index=['pid', 'Group', 'cm', 'idx', 'age', "edu_year", cog_item ,med_item], columns='net', values='rsq').reset_index()
            
            res = pg.mediation_analysis(data=df, x=med_item, m=list(yeo7_dict.keys()), y=cog_item,covar =["age","edu_year"], alpha=0.05,
                            seed=42)
            # display(res[res.sig=="Yes"])
            anyindirect = res.sig[-6:].to_list()
            if "Yes" in anyindirect:
                print(cm_name, tg_lab)
                display(res[res.sig=="Yes"])
                    

In [ ]:
stop

In [ ]:

# Get sub_info tbl
sub_info_tbl =pd.read_csv(f"{PROJ_HOME}/data/filt_sub_info.csv",encoding="gbk")
mean_edu_year = sub_info_tbl['edu_year'].mean().round()
# Fill education year nan to mean
sub_info_tbl['edu_year'].fillna(mean_edu_year, inplace=True)

# Get NAWM volume
nawm_tbl = pd.read_csv(f"{PROJ_HOME}/results/lms_stat/nawm_volume_raw.csv",names=["pid","nawm"])
nawm_tbl["lognawm"] = nawm_tbl["nawm"].apply(lambda x: np.log(x))
sub_info_tbl = pd.merge(sub_info_tbl, on="pid",right=nawm_tbl)

wmh_met_tbl = pd.read_csv(f"{PROJ_HOME}/results/lms_stat/2_wmh_dwi_metric.csv",names=["pid","dwim","val"])
wmh_met_pivot = wmh_met_tbl.pivot_table(index="pid",values="val",columns="dwim").reset_index()
sub_info_tbl = pd.merge(sub_info_tbl, on="pid",right=wmh_met_pivot)
# sub_info_tbl.columns

In [ ]:
med_items = ["rd", "fa","ad","adc","logwmh","lognawm"]
sum_res = pd.DataFrame()
for med_item in med_items:
    # Global mediation analysis
    for cog_item in ["门诊MMSE", "MoCA-merge"]:
        for TRACT_LABS in TRACT_LABS:
            for cm in CM_ARR[1:]:
                cm_lab, cm_idx, _ = cm
                # print(cog_item, tg_lab, cm_lab)
                df = pd.read_csv(f"{PROJ_HOME}/results/lr_glob_lm_cfc/{cm_lab}-cfc_{TRACT_LABS}.csv")
                df = df[df.idx ==cm_idx]

                df["age"] = df["pid"].map(SUB_ID_AGE_DICT)
                df["edu_year"] = df["pid"].map(sub_info_tbl.set_index("pid")["edu_year"].to_dict())
                df["cog"] = df["pid"].map(sub_info_tbl.set_index("pid")[cog_item].to_dict())
                df[med_item] = df["pid"].map(sub_info_tbl.set_index("pid")[med_item].to_dict())
                
                df = df[~np.isnan(df['cog'])]
                df= df[df[med_item]>0]
                pg_ma_res = pg.mediation_analysis(data=df, x=med_item, m='rsq', y='cog',covar =["age","edu_year"], alpha=0.05,
                                seed=42)
                if pg_ma_res.sig.to_list()[-1]=="Yes":
                    print(cog_item, cm_lab, TRACT_LABS)
                    pg_ma_res = pg_ma_res.assign(med=med_item, cog=cog_item, tg_lab=TRACT_LABS, cm=cm_lab, net="glob")
                    sum_res = pd.concat([sum_res, pg_ma_res])


    for cog_item in ["门诊MMSE", "MoCA-merge"]:
        for net_lab in yeo7_dict.keys():
            for TRACT_LABS in TRACT_LABS:
                net_df = pd.DataFrame()
                print(cog_item, net_lab, TRACT_LABS)
                for cm in CM_ARR[1:]:
                    
                    cm_name, cm_idx, _ = cm
                    
                    df = pd.read_csv(f"{PROJ_HOME}/results/lr_net_lm_cfc/{cm_name}-cfc_{TRACT_LABS}.csv")
                    df.cm = cm_name
                    df = df[df.net == net_lab][
                        [
                            "pid",
                            "Group",
                            "cm",
                            "rsq",
                        ]
                    ]
                    net_df = pd.concat([net_df, df], ignore_index=True)

                net_df = pd.merge(net_df, right=sub_info_tbl.set_index("pid")[["age", "edu_year", cog_item, med_item]],on="pid")
                # display(net_df)
                net_df= net_df[net_df[med_item]>0]
                net_pivot_df = net_df.pivot_table(index=['pid', 'Group', 'age',"edu_year", cog_item, med_item], columns='cm', values='rsq').reset_index()
                
                pg_ma_res = pg.mediation_analysis(data=net_pivot_df.copy(), x=med_item, m=CM_NAME_ARR, y=cog_item,covar =["age", "edu_year"], alpha=0.05,
                                seed=42)
                
                anyindirect = pg_ma_res.sig[-6:].to_list()
                if "Yes" in anyindirect:
                    print(net_lab, TRACT_LABS)
                    pg_ma_res = pg_ma_res.assign(med=med_item, cog=cog_item, tg_lab=TRACT_LABS, cm=cm_lab, net=net_lab)
                    sum_res = pd.concat([sum_res, pg_ma_res])
                    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

                    # 第一个图
                    sns.regplot(data=net_pivot_df, x=med_item, y=cog_item, ax=axes[0])
                    axes[0].set_title('Direct')

                    # 第二个图
                    sns.regplot(data=net_pivot_df, x=med_item, y="si-wei", ax=axes[1])
                    axes[1].set_title('X~med')

                    # 第三个图
                    sns.regplot(data=net_pivot_df, x="si-wei", y=cog_item, ax=axes[2])
                    axes[2].set_title('med~Y')

                    plt.tight_layout()  # 确保图表正确排列
                    plt.savefig(f"{PROJ_HOME}/results/{med_item}_{cog_item}_{TRACT_LABS}_{net_lab}_plwei.png")
sum_res.to_csv(f"{PROJ_HOME}/results/mediaction_{med_item}.csv")

## Sankey plot

In [ ]:
import plotly.graph_objects as go
# med_res_wmh_tbl = pd.read_csv(f"{PROJ_HOME}/results/med_wmh/mediaction_logwmh_wmh.csv",encoding="gbk")
# med_res_nawm_tbl = pd.read_csv(f"{PROJ_HOME}/results/med_nawm/mediaction_lognawm_nawm.csv",encoding="gbk")
# med_res_tbl = pd.concat([med_res_wmh_tbl, med_res_nawm_tbl])
# med_res_tbl=med_res_nawm_tbl
# med_res_tbl = med_res_tbl[med_res_tbl.med!="tfr"]

med_res_tbl = pd.read_csv(f"{PROJ_HOME}/results/med_wmh/mediaction_logwmh_wmh.csv", encoding="gbk")

In [ ]:
# Color
opaque_color = [
    "rgba(250, 127, 111, 1)",
    "rgba(255, 190, 122, 1)",
    "rgba(130, 176, 210, 1)"
]
trans_color=[
    "rgba(250, 127, 111, 0.3)",
    "rgba(255, 190, 122, 0.3)",
    "rgba(130, 176, 210, 0.3)"
]

In [ ]:

med_res_tbl =  med_res_tbl[~med_res_tbl.cog.isin(["DSS","AVLT(1+2+3)","Rey-O回忆","VFT"])]

# (med_res_tbl.sig=="Yes") & 
med_res_tbl = med_res_tbl[med_res_tbl.net!="glob"]
rep_item_arr = [["AVLT再认", "AVLT recall"],
                ["Rey-O再认", "Rey-O-recall"],
                ["MoCA-merge", "MoCA"],
                ["门诊MMSE", "MMSE"],
                ["Rey-O复制", "Rey-O-copy"],]
for rep_item in rep_item_arr:
    med_res_tbl.cog = med_res_tbl.cog.str.replace(*rep_item)
med_res_tbl.med = med_res_tbl.med.str.replace("rd", "RD")
med_res_tbl.med = med_res_tbl.med.str.replace("ad", "AD")
med_res_tbl.med = med_res_tbl.med.str.replace("fa", "FA")
med_res_tbl.med = med_res_tbl.med.str.replace("logwmh", "Volume")
med_res_tbl["medcog"] = med_res_tbl.apply(lambda x: x["med"] +"~"+ x["cog"], axis=1)

In [ ]:
cog_groups  = [
    [
        "MMSE",
        # "MoCA-merge",
    ],
    [
        "TMT(B-A)",],
    [
        "Stroop-TI",
        "Stroop-EI",
    ],
    [
        "DS(f+b)",
        "DS-forward",
        "DS-backward",
    ],
    [
        "AVLT4",
        "AVLT5",
        # "AVLT recall",
        "AVLT(1+2+3)",
    ],
    [
        "Rey-O回忆",
        "Rey-O-recall",
        "Rey-O-copy",
    ],
    [
        "BNT",
    ],
]


In [ ]:
for cog_group in cog_groups:
    sankey_tbl = med_res_tbl[med_res_tbl.cog.isin(cog_group)].copy()

    for j, TRACT_LABS in enumerate(TRACT_LABS[1:]):
        try:
            med_res_lm = sankey_tbl[sankey_tbl.tg_lab == TRACT_LABS]

            # display(med_res_lm[med_res_lm.path.str.contains("fg")])
            y_labs = med_res_lm.medcog.unique()
            x_labs = med_res_lm.med.unique()
            cm_by_net = []
            for cm in CM_NAME_ARR:
                for net in med_res_lm.net.unique():
                    cm_by_net.append(cm[:2].upper() + "-" + network_abbreviations[net])

            sankey_labs = [*x_labs, *cm_by_net, *y_labs, "Direct"]
            sankey_labs_color = [*[PLT3[1] for i in x_labs]]

            for cm in cm_by_net:
                for cm_cl in CM_ARR:
                    if cm[:2].lower() == cm_cl[0][:2]:

                        sankey_labs_color.append(cm_cl[2])

            sankey_labs_color = [*sankey_labs_color, *[PLT3[1] for i in y_labs]]
            # pos_x_arr = [*[0 for i in x_labs], *[0.5 for i in cm_by_net], *[1 for i in y_labs]]

            link_dict = dict(
                source=[],
                target=[],
                value=[],
                line=[],
                color=[],
            )
            node_dict = dict(
                pad=30,
                thickness=10,
                line=dict(color="black", width=0.0),
                label=sankey_labs,
                color=sankey_labs_color,
            )
            # based on each X item
            for j in range(len(med_res_lm) // 17):
                for index in range(5):
                    # Med~X
                    row = med_res_lm.iloc[17 * j + index]

                    cm_x_y = row["path"].split(" ~ ")

                    med_cm = cm_x_y[0]
                    link_dict["source"].append(sankey_labs.index(row["med"]))
                    link_dict["target"].append(
                        sankey_labs.index(
                            med_cm[:2].upper() + "-" + network_abbreviations[row["net"]]
                        )
                    )
                    link_dict["value"].append(1)

                    line_cl = "white" if row.sig == "Yes" else "rgba(0,0,0,0)"
                    link_dict["line"] = dict(color=line_cl, width=0)
                    # if med_res_lm.iloc[17 * j + 10].sig == "Yes":
                    if med_res_lm.iloc[17 * j + index + 5 + 7].sig == "Yes":
                        link_cl = opaque_color[0] if row.coef > 0 else opaque_color[2]
                    else:
                        if row.sig == "Yes":
                            link_cl = trans_color[0] if row.coef > 0 else trans_color[2]
                        else:
                            link_cl = "rgba(0,0,0, 0.03)"
                    # else:
                    #     link_cl = "rgba(255,255,255, 0.03)"
                    link_dict["color"].append(link_cl)

                    # ----------------------------------------------------------------
                    # Y~Med
                    row = med_res_lm.iloc[17 * j + index + 5]
                    print(row.path)
                    cm_x_y = row["path"].split(" ~ ")

                    med_cm = cm_x_y[1]
                    link_dict["source"].append(
                        sankey_labs.index(
                            med_cm[:2].upper() + "-" + network_abbreviations[row["net"]]
                        )
                    )
                    link_dict["target"].append(sankey_labs.index(row["medcog"]))

                    link_dict["value"].append(1)

                    line_cl = "white" if row.sig == "Yes" else "rgba(0,0,0,0)"
                    link_dict["line"] = dict(color=line_cl, width=0)

                    # if med_res_lm.iloc[17 * j + 10].sig == "Yes":
                    print(med_res_lm.iloc[17 * j + index + 5 + 7].path)
                    if med_res_lm.iloc[17 * j + index + 5 + 7].sig == "Yes":
                        link_cl = opaque_color[0] if row.coef > 0 else opaque_color[2]
                    else:
                        if row.sig == "Yes":
                            link_cl = trans_color[0] if row.coef > 0 else trans_color[2]
                        else:
                            link_cl = "rgba(0,0,0, 0.03)"
                    # else:
                    #     link_cl = "rgba(255,255,255, 0.03)"

                    link_dict["color"].append(link_cl)

                # Direct
                row = med_res_lm.iloc[17 * j + 11]
                link_dict["source"].append(sankey_labs.index(row["med"]))
                link_dict["target"].append(sankey_labs.index("Direct"))
                link_dict["value"].append(1)
                if med_res_lm.iloc[17 * j + 11].sig == "Yes":
                    link_cl = opaque_color[0] if row.coef > 0 else opaque_color[2]
                else:
                    link_cl = "rgba(0,0,0, 0.03)"
                link_dict["color"].append(link_cl)

                row = med_res_lm.iloc[17 * j + 11]
                link_dict["source"].append(sankey_labs.index("Direct"))
                link_dict["target"].append(sankey_labs.index(row["medcog"]))
                link_dict["value"].append(1)
                if med_res_lm.iloc[17 * j + 11].sig == "Yes":
                    link_cl = opaque_color[0] if row.coef > 0 else opaque_color[2]

                else:
                    link_cl = "rgba(0,0,0, 0.03)"

                link_dict["color"].append(link_cl)

            fig = go.Figure(data=[go.Sankey(node=node_dict, link=link_dict)])
            
            fig.update_layout(title_text=TRACT_LABS.upper(), font_size=15)

            fig.show()
            if not os.path.exists(f"{PROJ_HOME}/figures/MA_{cog_group[0]}"):
                os.makedirs(f"{PROJ_HOME}/figures/MA_{cog_group[0]}")
            fig.write_image(
                f"{PROJ_HOME}/figures/MA_{cog_group[0]}/sankey-{TRACT_LABS}-{row.med}-{cog_group[0]}.svg",
                format="svg",
                scale=1,
            )
        except:
            pass

Y ~ co-wei
Indirect co-wei
Y ~ fg-wei
Indirect fg-wei
Y ~ pt-wei
Indirect pt-wei
Y ~ si-wei
Indirect si-wei
Y ~ pl-wei
Indirect pl-wei
